In [1]:
from sentimentLDA import *
from gensim.models import Word2Vec
import os
import urllib
import tarfile
import dill
import pandas as pd
from tqdm import tqdm
from nltk.corpus import sentiwordnet as swn
import numpy as np
tqdm.pandas()
vocabSize = 50000

C:\Libraries\anaconda3\lib\site-packages\tqdm\std.py:697: FutureWarning: The Panel class is removed from pandas. Accessing it from the top-level namespace will also be removed in the next version
  from pandas import Panel


In [2]:
def chunks(lst, n):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

In [3]:
word_vectors = Word2Vec.load("./sentiment140-word2vec.model").wv

In [4]:
dataframe = pd.read_csv("./sentiment140-cleaned.csv", encoding="ISO-8859-1", engine="python")

In [5]:
dataframe.head()

,sentiment,user,text
0,0,_TheSpecialOne_,@switchfoot awww that is a bummer you shoulda ...
1,0,scotthamilton,is upset that he can t update his facebook by ...
2,0,mattycus,@kenichan i dived many times for the ball mana...
3,0,ElleCTF,my whole body feels itchy and like its on fire
4,0,Karoli,@nationwideclass no it is not behaving at all ...


In [6]:
n=1600
df = dataframe.sample(n=n, random_state=231)
df = df.reset_index(drop=True)

In [7]:
cluster_good = ["good", "nice", "cool", "lovely", "wonderful", "great", "awesome", "fantastic", "amazing", "fun", "excellent"]
cluster_bad = ["bad", "horrible", "terrible", "awful", "worst", "shitty", "crappy", "sucks", "hate"]
def sentiFun(word):
    if word in word_vectors.vocab:
        posScore = 1 - np.average(word_vectors.distances(word, cluster_good))
        negScore = 1 - np.average(word_vectors.distances(word, cluster_bad))
        return (posScore, negScore)
    return (0, 0)

In [8]:
def sentiFun(word):
    synsets = swn.senti_synsets(word)
    posScore = np.mean([s.pos_score() for s in synsets])
    negScore = np.mean([s.neg_score() for s in synsets])
    return (posScore, negScore)

In [36]:
#nTopics, alpha, beta, gamma
sampler = SentimentLDAGibbsSampler(10, 2.5, 0.1, 0.3, seed=231)

In [37]:
sampler.run(tqdm(df['text'].tolist()), sentiFun, 200, "./lda.dll", True)

100%|██████████| 1600/1600 [00:00<00:00, 2629.02it/s]
C:\Libraries\anaconda3\lib\site-packages\numpy\core\fromnumeric.py:3372: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Libraries\anaconda3\lib\site-packages\numpy\core\_methods.py:170: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 200/200 [02:15<00:00,  1.47it/s]


print("Top discriminative words for topic t and sentiment s ie words v for which p(v | t, s) is maximum")
lists = sampler.getTopKWords(25)
for lst in lists:
    (t, s, words) = lst
    print("  Topic: {} Sentiment: {}".format(t,s))
    for cnk in chunks(words, 5):
        print("    "+ ", ".join(cnk))
    print(" ")

sampler.conditionalDistribution(0, 0)

In [38]:
topicSentimentList = sampler.getTopKWords(25)

In [39]:
def getTweetSentiment(tweetIndex, words):
    probabilities = []
    for wordIndex in range(len(words)):
        probabilities.append(sampler.conditionalDistribution(tweetIndex, wordIndex))
    probabilities = np.mean(probabilities, axis=0)
    topic = 0
    topicProb = 0
    for i in range(len(probabilities)):
        sentiments = probabilities[i]
        probability = sentiments[0] + sentiments[1]
        if probability > topicProb:
            topicProb = probability
            topic = i
    (_, _, topicClusterGood) = topicSentimentList[i*2 + 1]
    (_, _, topicClusterBad)  = topicSentimentList[i*2]
    topicClusterGood = [x for x in topicClusterGood if x in word_vectors.vocab]
    topicClusterBad  = [x for x in topicClusterBad if x in word_vectors.vocab]
    wordvectors = []
    for word in words:
        if not (word in word_vectors.vocab):
            continue
        wordvectors.append(word_vectors[word])
    if len(wordvectors) == 0:
        return 0
    sentencevector = np.mean(wordvectors, axis=0)

    positive = 1-np.average(word_vectors.distances(sentencevector, cluster_good))
    negative = 1-np.average(word_vectors.distances(sentencevector, cluster_bad))
    topicPositive = 1-np.average(word_vectors.distances(sentencevector, topicClusterGood))
    topicNegative = 1-np.average(word_vectors.distances(sentencevector, topicClusterBad))
    return positive - negative + (topicPositive - topicNegative)

#50% accuracy..., 55.4%
def getTweetSentiment(tweetIndex, words):
    sentiment = 0
    for wordIndex in range(words):
        probabilityMatrix = sampler.conditionalDistribution(tweetIndex, wordIndex)
        topic = 0
        topicProb = 0
        for i in range(len(probabilityMatrix)):
            sentiments = probabilityMatrix[i]
            probability = sentiments[0] + sentiments[1]
            if probability > topicProb:
                topicProb = probability
                topic = i
        
        sentiments = probabilityMatrix[topic]
        sentiment -= sentiments[0]
        sentiment += sentiments[1]
    return sentiment

#51% 57,7%
def getTweetSentiment(tweetIndex, words):
    sentiment = 0
    for wordIndex in range(words):
        probabilityMatrix = sampler.conditionalDistribution(tweetIndex, wordIndex)
        for sentiments in probabilityMatrix:
            sentiment -= sentiments[0]
            sentiment += sentiments[1]

    return sentiment

#48% #55.5%
def getTweetSentiment(tweetIndex, words):
    sentiment = 0
    for wordIndex in range(words):
        probabilityMatrix = sampler.conditionalDistribution(tweetIndex, wordIndex)
        for sentiments in probabilityMatrix:
            if(sentiments[0] > sentiments[1]):
                sentiment -= 1
            else:
                sentiment += 1

    return sentiment

In [40]:
df['predicted'] = df.progress_apply(lambda row: getTweetSentiment(row.name, row['text'].split()), axis=1)
df['predicted'] = df.progress_apply(lambda row: 4 if row['predicted'] > 0 else 0, axis=1)
df['predict_correct'] = df.progress_apply(lambda row: row['sentiment'] == row['predicted'], axis=1)

100%|██████████| 1600/1600 [00:00<00:00, 95055.05it/s]


In [41]:
print(df['predict_correct'].value_counts()[True] / n)

0.665625
